# 🇻🇳 Vietnamese Fake News Detection: Crawling & Preprocessing Pipeline

This notebook combines web crawling and data preprocessing for Vietnamese fake news detection.

## 📋 Overview

1. **Crawling**: Extract news articles from Vietnamese news websites
2. **Preprocessing**: Process text and image data for machine learning
3. **Dataset Creation**: Create structured datasets ready for training

## 🔧 Dependencies

- Transformers (PhoBERT for Vietnamese text)
- Torchvision (ResNet for image processing)
- BeautifulSoup (Web scraping)
- PIL (Image processing)


In [ ]:
# Install required packages if not already installed
# !pip install transformers torch torchvision Pillow beautifulsoup4 lxml httpx tqdm nest_asyncio

import sys
import os
import json
import asyncio
import nest_asyncio
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple
import torch
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

# Apply nest_asyncio for notebook compatibility
nest_asyncio.apply()

# Add project root to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

print(f"✅ Project root: {project_root}")
print(f"✅ Python path updated")

## 🕷️ Part 1: Web Crawling

### Configuration

Set up crawling parameters and target websites


In [ ]:
# Crawling Configuration
CRAWLING_CONFIG = {
    "max_pages": 10,  # Number of pages to crawl
    "max_depth": 1,  # Maximum depth for deep crawling
    "save_format": ".json",  # Output format
    "output_dir": "./crawled_data",  # Output directory
    "image_download": True,  # Whether to download images
    "delay_between_requests": 1,  # Delay in seconds
}

# Target Vietnamese news websites
TARGET_SITES = {
    "vnexpress": {
        "base_url": "https://vnexpress.net/",
        "categories": ["thoi-su", "the-gioi", "kinh-doanh", "giai-tri"],
        "crawler_class": "VnExpressCrawler",
    },
    "thanhnien": {
        "base_url": "https://thanhnien.vn/",
        "categories": ["thoisu", "thegioi", "kinhdoanh", "vanhoa"],
        "crawler_class": "ThanhNienCrawler",
    },
}

print("🔧 Crawling configuration loaded:")
for key, value in CRAWLING_CONFIG.items():
    print(f"  • {key}: {value}")

print("\n📰 Target websites:")
for site, config in TARGET_SITES.items():
    print(f"  • {site}: {config['base_url']}")

In [ ]:
# Create output directories
output_dir = Path(CRAWLING_CONFIG["output_dir"])
output_dir.mkdir(exist_ok=True)

# Create subdirectories for each site
for site_name in TARGET_SITES.keys():
    site_dir = output_dir / site_name
    site_dir.mkdir(exist_ok=True)
    (site_dir / "images").mkdir(exist_ok=True)

print(f"📁 Created output directories in: {output_dir}")

In [ ]:
async def crawl_site(site_name: str, site_config: Dict) -> Dict:
    """
    Crawl a specific news website
    """
    try:
        # Import the appropriate crawler
        if site_config["crawler_class"] == "VnExpressCrawler":
            from src.crawler.news.real.VnExpressCrawler import VnExpressCrawler

            crawler = VnExpressCrawler()
        elif site_config["crawler_class"] == "ThanhNienCrawler":
            from src.crawler.news.real.ThanhNienCrawler import ThanhNienCrawler

            crawler = ThanhNienCrawler()
        else:
            print(f"❌ Unknown crawler class: {site_config['crawler_class']}")
            return {"error": "Unknown crawler class"}

        print(f"🕷️ Starting to crawl {site_name}...")

        # Set output paths
        site_output_dir = output_dir / site_name
        json_path = site_output_dir / f"{site_name}_articles.json"

        # Crawl with file saving
        await crawler.arun(
            save_to_file=True,
            save_format=CRAWLING_CONFIG["save_format"],
            max_pages=CRAWLING_CONFIG["max_pages"],
        )

        # Extract structured data
        articles = await crawler.extract_with_schema()

        # Save structured data
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(articles, f, ensure_ascii=False, indent=2)

        result = {
            "site": site_name,
            "articles_count": len(articles) if articles else 0,
            "output_file": str(json_path),
            "timestamp": datetime.now().isoformat(),
        }

        print(f"✅ Crawled {result['articles_count']} articles from {site_name}")
        return result

    except Exception as e:
        print(f"❌ Error crawling {site_name}: {str(e)}")
        return {"site": site_name, "error": str(e)}


# Test crawling function
print("🔧 Crawling function defined successfully")

In [ ]:
{
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import preprocessing modules\n",
    "try:\n",
    "    from src.preprocessing import CombinedPreprocessor\n",
    "    print(\"✅ Successfully imported CombinedPreprocessor\")\n",
    "except ImportError as e:\n",
    "    print(f\"❌ Error importing preprocessing modules: {e}\")\n",
    "    print(\"Make sure the preprocessing module is properly installed\")\n",
    "    # Alternative import path\n",
    "    try:\n",
    "        sys.path.append('./src')\n",
    "        from preprocessing.combined_preprocessing import CombinedPreprocessor\n",
    "        print(\"✅ Successfully imported CombinedPreprocessor (alternative path)\")\n",
    "    except ImportError as e2:\n",
    "        print(f\"❌ Still cannot import: {e2}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize the preprocessor\n",
    "try:\n",
    "    preprocessor = CombinedPreprocessor(\n",
    "        text_model_name=PREPROCESSING_CONFIG[\"text_model\"],\n",
    "        image_model_name=PREPROCESSING_CONFIG[\"image_model\"],\n",
    "        language=PREPROCESSING_CONFIG[\"language\"],\n",
    "        max_text_length=PREPROCESSING_CONFIG[\"max_text_length\"],\n",
    "        device=device\n",
    "    )\n",
    "    print(\"✅ Preprocessor initialized successfully!\")\n",
    "    print(f\"  • Text model: {PREPROCESSING_CONFIG['text_model']}\")\n",
    "    print(f\"  • Image model: {PREPROCESSING_CONFIG['image_model']}\")\n",
    "    print(f\"  • Device: {device}\")\n",
    "except Exception as e:\n",
    "    print(f\"❌ Error initializing preprocessor: {e}\")\n",
    "    print(\"This might be due to missing dependencies or model downloads\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Load Crawled Data\n",
    "Load the crawled data for preprocessing"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def load_crawled_data(data_dir: str = \"./crawled_data\") -> List[Dict]:\n",
    "    \"\"\"\n",
    "    Load all crawled JSON files from the data directory\n",
    "    \"\"\"\n",
    "    all_articles = []\n",
    "    data_path = Path(data_dir)\n",
    "    \n",
    "    if not data_path.exists():\n",
    "        print(f\"❌ Data directory not found: {data_path}\")\n",
    "        return []\n",
    "    \n",
    "    # Find all JSON files\n",
    "    json_files = list(data_path.glob(\"**/*.json\"))\n",
    "    \n",
    "    for json_file in json_files:\n",
    "        if json_file.name == \"crawling_summary.json\":\n",
    "            continue  # Skip summary file\n",
    "        \n",
    "        try:\n",
    "            with open(json_file, 'r', encoding='utf-8') as f:\n",
    "                articles = json.load(f)\n",
    "            \n",
    "            # Add source information\n",
    "            for article in articles:\n",
    "                article['source_file'] = str(json_file)\n",
    "                article['source_site'] = json_file.parent.name\n",
    "            \n",
    "            all_articles.extend(articles)\n",
    "            print(f\"✅ Loaded {len(articles)} articles from {json_file.name}\")\n",
    "            \n",
    "        except Exception as e:\n",
    "            print(f\"❌ Error loading {json_file}: {e}\")\n",
    "    \n",
    "    print(f\"\\n📊 Total articles loaded: {len(all_articles)}\")\n",
    "    return all_articles\n",
    "\n",
    "# Load crawled data\n",
    "crawled_articles = load_crawled_data()\n",
    "\n",
    "if crawled_articles:\n",
    "    # Display sample article\n",
    "    print(\"\\n📄 Sample article structure:\")\n",
    "    sample = crawled_articles[0]\n",
    "    for key, value in sample.items():\n",
    "        if isinstance(value, str) and len(value) > 100:\n",
    "            print(f\"  • {key}: {value[:100]}...\")\n",
    "        else:\n",
    "            print(f\"  • {key}: {value}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Data Preparation\n",
    "Prepare data for preprocessing by extracting text, labels, and image paths"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def prepare_data_for_preprocessing(articles: List[Dict]) -> Tuple[List[str], List[str], List[int]]:\n",
    "    \"\"\"\n",
    "    Prepare crawled data for preprocessing\n",
    "    \n",
    "    Returns:\n",
    "        Tuple of (texts, image_paths, labels)\n",
    "    \"\"\"\n",
    "    texts = []\n",
    "    image_paths = []\n",
    "    labels = []\n",
    "    \n",
    "    for i, article in enumerate(articles):\n",
    "        # Extract text content\n",
    "        text = article.get('text', '') or article.get('content', '') or article.get('title', '')\n",
    "        \n",
    "        if text:\n",
    "            texts.append(text)\n",
    "            \n",
    "            # Extract image path\n",
    "            images = article.get('images', [])\n",
    "            if images and len(images) > 0:\n",
    "                # Handle different image formats\n",
    "                if isinstance(images[0], str):\n",
    "                    image_path = images[0]\n",
    "                elif isinstance(images[0], dict):\n",
    "                    image_path = images[0].get('url', '') or images[0].get('path', '')\n",
    "                else:\n",
    "                    image_path = str(images[0])\n",
    "                \n",
    "                image_paths.append(image_path)\n",
    "            else:\n",
    "                # Create placeholder for missing images\n",
    "                image_paths.append(None)\n",
    "            \n",
    "            # Extract or create label\n",
    "            # For crawled data, we might not have labels, so use default\n",
    "            label = article.get('label', article.get('is_fake', 0))\n",
    "            if isinstance(label, str):\n",
    "                # Convert string labels to integers\n",
    "                label = 1 if label.lower() in ['fake', 'false', '0'] else 0\n",
    "            labels.append(int(label))\n",
    "    \n",
    "    return texts, image_paths, labels\n",
    "\n",
    "# Prepare data\n",
    "if crawled_articles:\n",
    "    texts, image_paths, labels = prepare_data_for_preprocessing(crawled_articles)\n",
    "    \n",
    "    print(f\"📊 Data preparation completed:\")\n",
    "    print(f\"  • Text samples: {len(texts)}\")\n",
    "    print(f\"  • Image paths: {len(image_paths)}\")\n",
    "    print(f\"  • Labels: {len(labels)}\")\n",
    "    print(f\"  • Valid images: {sum(1 for path in image_paths if path is not None)}\")\n",
    "    print(f\"  • Label distribution: {np.bincount(labels)}\")\n",
    "else:\n",
    "    print(\"❌ No crawled data available for preparation\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Preprocessing Execution\n",
    "Run the preprocessing pipeline on the prepared data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def create_placeholder_images(image_paths: List[str], split_name: str = \"crawled\") -> List[str]:\n",
    "    \"\"\"\n",
    "    Create placeholder images for missing image paths\n",
    "    \"\"\"\n",
    "    placeholder_dir = Path(f\"./placeholder_images/{split_name}\")\n",
    "    placeholder_dir.mkdir(parents=True, exist_ok=True)\n",
    "    \n",
    "    processed_paths = []\n",
    "    \n",
    "    for i, image_path in enumerate(image_paths):\n",
    "        if image_path is None or not os.path.exists(image_path):\n",
    "            # Create placeholder image\n",
    "            placeholder_path = placeholder_dir / f\"placeholder_{i}.jpg\"\n",
    "            \n",
    "            if not placeholder_path.exists():\n",
    "                from PIL import Image\n",
    "                placeholder_array = np.random.randint(128, 200, (224, 224, 3), dtype=np.uint8)\n",
    "                placeholder_image = Image.fromarray(placeholder_array)\n",
    "                placeholder_image.save(placeholder_path)\n",
    "            \n",
    "            processed_paths.append(str(placeholder_path))\n",
    "        else:\n",
    "            processed_paths.append(image_path)\n",
    "    \n",
    "    return processed_paths\n",
    "\n",
    "# Process crawled data\n",
    "if crawled_articles and 'preprocessor' in locals():\n",
    "    try:\n",
    "        print(\"🔄 Starting preprocessing...\")\n",
    "        \n",
    "        # Create placeholder images for missing ones\n",
    "        processed_image_paths = create_placeholder_images(image_paths)\n",
    "        \n",
    "        # Create output directory\n",
    "        output_dir = \"./processed_data/crawled\"\n",
    "        os.makedirs(output_dir, exist_ok=True)\n",
    "        \n",
    "        # Run preprocessing\n",
    "        dataset, metadata = preprocessor.preprocess_dataset(\n",
    "            texts=texts,\n",
    "            image_paths=processed_image_paths,\n",
    "            labels=labels,\n",
    "            save_dir=output_dir,\n",
    "            save_format=PREPROCESSING_CONFIG[\"save_format\"],\n",
    "            batch_size=PREPROCESSING_CONFIG[\"batch_size\"]\n",
    "        )\n",
    "        \n",
    "        print(\"✅ Preprocessing completed successfully!\")\n",
    "        print(f\"📁 Processed data saved to: {output_dir}\")\n",
    "        print(f\"📊 Dataset size: {len(dataset)} samples\")\n",
    "        print(f\"📝 Text features shape: {metadata['text_feature_shape']}\")\n",
    "        print(f\"🖼️ Image features shape: {metadata['image_feature_shape']}\")\n",
    "        \n",
    "    except Exception as e:\n",
    "        print(f\"❌ Error during preprocessing: {e}\")\n",
    "        import traceback\n",
    "        traceback.print_exc()\n",
    "else:\n",
    "    print(\"❌ Cannot run preprocessing - missing data or preprocessor\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📊 Part 3: Dataset Analysis & Visualization\n",
    "\n",
    "Analyze the processed dataset and create visualizations"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from collections import Counter\n",
    "\n",
    "# Set style\n",
    "plt.style.use('seaborn-v0_8')\n",
    "sns.set_palette(\"husl\")\n",
    "\n",
    "def analyze_dataset(metadata_path: str = \"./processed_data/crawled/metadata.json\"):\n",
    "    \"\"\"\n",
    "    Analyze the processed dataset\n",
    "    \"\"\"\n",
    "    try:\n",
    "        with open(metadata_path, 'r') as f:\n",
    "            metadata = json.load(f)\n",
    "        \n",
    "        print(\"📊 Dataset Analysis:\")\n",
    "        print(\"=\" * 40)\n",
    "        for key, value in metadata.items():\n",
    "            print(f\"  • {key}: {value}\")\n",
    "        \n",
    "        return metadata\n",
    "        \n",
    "    except FileNotFoundError:\n",
    "        print(f\"❌ Metadata file not found: {metadata_path}\")\n",
    "        return None\n",
    "    except Exception as e:\n",
    "        print(f\"❌ Error analyzing dataset: {e}\")\n",
    "        return None\n",
    "\n",
    "# Analyze dataset\n",
    "metadata = analyze_dataset()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def visualize_text_statistics(texts: List[str]):\n",
    "    \"\"\"\n",
    "    Visualize text statistics\n",
    "    \"\"\"\n",
    "    if not texts:\n",
    "        print(\"❌ No texts to visualize\")\n",
    "        return\n",
    "    \n",
    "    # Calculate text lengths\n",
    "    text_lengths = [len(text.split()) for text in texts]\n",
    "    \n",
    "    fig, axes = plt.subplots(2, 2, figsize=(15, 10))\n",
    "    fig.suptitle('📝 Text Statistics Analysis', fontsize=16, fontweight='bold')\n",
    "    \n",
    "    # Text length distribution\n",
    "    axes[0, 0].hist(text_lengths, bins=30, alpha=0.7, edgecolor='black')\n",
    "    axes[0, 0].set_title('Text Length Distribution')\n",
    "    axes[0, 0].set_xlabel('Number of Words')\n",
    "    axes[0, 0].set_ylabel('Frequency')\n",
    "    axes[0, 0].axvline(np.mean(text_lengths), color='red', linestyle='--', label=f'Mean: {np.mean(text_lengths):.1f}')\n",
    "    axes[0, 0].legend()\n",
    "    \n",
    "    # Box plot of text lengths\n",
    "    axes[0, 1].boxplot(text_lengths)\n",
    "    axes[0, 1].set_title('Text Length Box Plot')\n",
    "    axes[0, 1].set_ylabel('Number of Words')\n",
    "    \n",
    "    # Word frequency (top 20)\n",
    "    all_words = ' '.join(texts).lower().split()\n",
    "    word_freq = Counter(all_words)\n",
    "    top_words = word_freq.most_common(20)\n",
    "    \n",
    "    words, counts = zip(*top_words)\n",
    "    axes[1, 0].barh(range(len(words)), counts)\n",
    "    axes[1, 0].set_yticks(range(len(words)))\n",
    "    axes[1, 0].set_yticklabels(words, fontsize=8)\n",
    "    axes[1, 0].set_title('Top 20 Most Common Words')\n",
    "    axes[1, 0].set_xlabel('Frequency')\n",
    "    \n",
    "    # Text length statistics\n",
    "    stats_text = f\"\"\"\n",
    "    Total Texts: {len(texts)}\n",
    "    Mean Length: {np.mean(text_lengths):.1f} words\n",
    "    Median Length: {np.median(text_lengths):.1f} words\n",
    "    Std Dev: {np.std(text_lengths):.1f} words\n",
    "    Min Length: {min(text_lengths)} words\n",
    "    Max Length: {max(text_lengths)} words\n",
    "    \"\"\"\n",
    "    axes[1, 1].text(0.1, 0.5, stats_text, fontsize=12, verticalalignment='center')\n",
    "    axes[1, 1].set_title('Text Statistics Summary')\n",
    "    axes[1, 1].axis('off')\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "\n",
    "# Visualize text statistics\n",
    "if 'texts' in locals() and texts:\n",
    "    visualize_text_statistics(texts)\n",
    "else:\n",
    "    print(\"❌ No texts available for visualization\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def visualize_label_distribution(labels: List[int]):\n",
    "    \"\"\"\n",
    "    Visualize label distribution\n",
    "    \"\"\"\n",
    "    if not labels:\n",
    "        print(\"❌ No labels to visualize\")\n",
    "        return\n",
    "    \n",
    "    label_counts = Counter(labels)\n",
    "    \n",
    "    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))\n",
    "    fig.suptitle('📊 Label Distribution Analysis', fontsize=16, fontweight='bold')\n",
    "    \n",
    "    # Bar chart\n",
    "    label_names = ['Real News', 'Fake News']\n",
    "    colors = ['green', 'red']\n",
    "    \n",
    "    bars = ax1.bar(label_names, [label_counts.get(0, 0), label_counts.get(1, 0)], color=colors, alpha=0.7)\n",
    "    ax1.set_title('Label Distribution')\n",
    "    ax1.set_ylabel('Count')\n",
    "    \n",
    "    # Add count labels on bars\n",
    "    for bar, count in zip(bars, [label_counts.get(0, 0), label_counts.get(1, 0)]):\n",
    "        height = bar.get_height()\n",
    "        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.1,\n",
    "                f'{count}', ha='center', va='bottom', fontweight='bold')\n",
    "    \n",
    "    # Pie chart\n",
    "    sizes = [label_counts.get(0, 0), label_counts.get(1, 0)]\n",
    "    if sum(sizes) > 0:\n",
    "        ax2.pie(sizes, labels=label_names, colors=colors, autopct='%1.1f%%', startangle=90)\n",
    "        ax2.set_title('Label Proportion')\n",
    "    else:\n",
    "        ax2.text(0.5, 0.5, 'No data available', ha='center', va='center', transform=ax2.transAxes)\n",
    "        ax2.set_title('Label Proportion')\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "    \n",
    "    # Print statistics\n",
    "    print(f\"📊 Label Statistics:\")\n",
    "    print(f\"  • Total samples: {len(labels)}\")\n",
    "    print(f\"  • Real news (0): {label_counts.get(0, 0)} ({label_counts.get(0, 0)/len(labels)*100:.1f}%)\")\n",
    "    print(f\"  • Fake news (1): {label_counts.get(1, 0)} ({label_counts.get(1, 0)/len(labels)*100:.1f}%)\")\n",
    "\n",
    "# Visualize label distribution\n",
    "if 'labels' in locals() and labels:\n",
    "    visualize_label_distribution(labels)\n",
    "else:\n",
    "    print(\"❌ No labels available for visualization\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 💾 Part 4: Export & Save\n",
    "\n",
    "Export processed data in various formats for downstream use"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def export_processed_data(\n",
    "    output_dir: str = \"./processed_data/crawled\",\n",
    "    export_formats: List[str] = [\"pkl\", \"npz\", \"csv\"]\n",
    "):\n",
    "    \"\"\"\n",
    "    Export processed data in multiple formats\n",
    "    \"\"\"\n",
    "    try:\n",
    "        # Load combined dataset\n",
    "        combined_path = os.path.join(output_dir, \"combined_dataset.pkl\")\n",
    "        \n",
    "        if os.path.exists(combined_path):\n",
    "            text_features, image_features, labels = preprocessor.load_combined_dataset(combined_path)\n",
    "            \n",
    "            print(f\"📦 Exporting processed data...\")\n",
    "            \n",
    "            # Export as NumPy compressed\n",
    "            if \"npz\" in export_formats:\n",
    "                npz_path = os.path.join(output_dir, \"dataset.npz\")\n",
    "                np.savez(\n",
    "                    npz_path,\n",
    "                    text_features=text_features,\n",
    "                    image_features=image_features,\n",
    "                    labels=labels\n",
    "                )\n",
    "                print(f\"  ✅ Exported as NPZ: {npz_path}\")\n",
    "            \n",
    "            # Export as CSV (flattened features)\n",
    "            if \"csv\" in export_formats:\n",
    "                csv_path = os.path.join(output_dir, \"dataset_metadata.csv\")\n",
    "                \n",
    "                # Create metadata CSV\n",
    "                metadata_df = pd.DataFrame({\n",
    "                    'sample_id': range(len(labels)),\n",
    "                    'label': labels,\n",
    "                    'text_feature_shape': [text_features[i].shape for i in range(len(text_features))],\n",
    "                    'image_feature_shape': [image_features[i].shape for i in range(len(image_features))]\n",
    "                })\n",
    "                \n",
    "                metadata_df.to_csv(csv_path, index=False)\n",
    "                print(f\"  ✅ Exported metadata as CSV: {csv_path}\")\n",
    "            \n",
    "            print(f\"\\n✅ Export completed! Files saved in: {output_dir}\")\n",
    "            \n",
    "        else:\n",
    "            print(f\"❌ Combined dataset not found: {combined_path}\")\n",
    "            \n",
    "    except Exception as e:\n",
    "        print(f\"❌ Error exporting data: {e}\")\n",
    "        import traceback\n",
    "        traceback.print_exc()\n",
    "\n",
    "# Export processed data\n",
    "export_processed_data()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 🎯 Summary & Next Steps\n",
    "\n",
    "### What was accomplished:\n",
    "1. ✅ **Web Crawling**: Set up crawling configuration for Vietnamese news sites\n",
    "2. ✅ **Data Preprocessing**: Configured PhoBERT and ResNet for multimodal processing\n",
    "3. ✅ **Pipeline Integration**: Combined crawling and preprocessing in one workflow\n",
    "4. ✅ **Visualization**: Added analysis and visualization tools\n",
    "5. ✅ **Export**: Multiple format export options\n",
    "\n",
    "### How to use this notebook:\n",
    "1. **Run crawling**: Uncomment the `crawling_results = await run_crawling()` line\n",
    "2. **Process data**: The preprocessing will automatically run after crawling\n",
    "3. **Analyze results**: Use the visualization cells to understand your dataset\n",
    "4. **Export data**: Use the export function to save in your preferred format\n",
    "\n",
    "### Next steps for your project:\n",
    "1. **Training**: Use the processed data to train your COOLANT model\n",
    "2. **Validation**: Split data and create validation sets\n",
    "3. **Model tuning**: Experiment with different preprocessing parameters\n",
    "4. **Deployment**: Integrate this pipeline into your main workflow"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Quick summary function\n",
    "def print_pipeline_summary():\n",
    "    \"\"\"\n",
    "    Print a summary of the pipeline status\n",
    "    \"\"\"\n",
    "    print(\"🚀 Vietnamese Fake News Detection Pipeline Summary\")\n",
    "    print(\"=\" * 60)\n",
    "    \n",
    "    # Check crawled data\n",
    "    crawled_dir = Path(\"./crawled_data\")\n",
    "    if crawled_dir.exists():\n",
    "        json_files = list(crawled_dir.glob(\"**/*.json\"))\n",
    "        print(f\"📰 Crawled data: {len(json_files)} JSON files found\")\n",
    "    else:\n",
    "        print(\"📰 Crawled data: Not found\")\n",
    "    \n",
    "    # Check processed data\n",
    "    processed_dir = Path(\"./processed_data/crawled\")\n",
    "    if processed_dir.exists():\n",
    "        files = list(processed_dir.glob(\"*\"))\n",
    "        print(f\"🔄 Processed data: {len(files)} files found\")\n",
    "    else:\n",
    "        print(\"🔄 Processed data: Not found\")\n",
    "    \n",
    "    # Check preprocessor\n",
    "    if 'preprocessor' in locals():\n",
    "        print(\"✅ Preprocessor: Initialized\")\n",
    "    else:\n",
    "        print(\"❌ Preprocessor: Not initialized\")\n",
    "    \n",
    "    print(\"\\n🎯 Ready to:\")\n",
    "    print(\"  1. Run web crawling (uncomment crawling line)\")\n",
    "    print(\"  2. Process crawled data with PhoBERT + ResNet\")\n",
    "    print(\"  3. Analyze and visualize results\")\n",
    "    print(\"  4. Export for model training\")\n",
    "\n",
    "# Print summary\n",
    "print_pipeline_summary()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}

In [ ]:
# Execute crawling for all target sites
async def run_crawling():
    """
    Run crawling for all configured websites
    """
    print("🚀 Starting web crawling...")
    print("=" * 50)

    crawling_results = []

    for site_name, site_config in TARGET_SITES.items():
        print(f"\n📰 Processing {site_name}...")
        result = await crawl_site(site_name, site_config)
        crawling_results.append(result)

        # Add delay between requests
        await asyncio.sleep(CRAWLING_CONFIG["delay_between_requests"])

    # Save crawling summary
    summary_path = output_dir / "crawling_summary.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(crawling_results, f, ensure_ascii=False, indent=2)

    print("\n" + "=" * 50)
    print("✅ Crawling completed!")

    # Print summary
    total_articles = sum(r.get("articles_count", 0) for r in crawling_results)
    print(f"📊 Total articles crawled: {total_articles}")
    print(f"📁 Summary saved to: {summary_path}")

    return crawling_results


# Run crawling
# Uncomment the line below to execute crawling
# crawling_results = await run_crawling()

## 🔄 Part 2: Data Preprocessing

### Configuration

Set up preprocessing parameters for text and image processing


In [ ]:
# Preprocessing Configuration
PREPROCESSING_CONFIG = {
    "text_model": "vinai/phobert-base",  # Vietnamese PhoBERT
    "image_model": "resnet18",  # ResNet for images
    "language": "vi",  # Vietnamese
    "max_text_length": 512,
    "batch_size": 16,  # Adjust based on memory
    "save_format": "pkl",  # 'pkl' or 'npz'
    "image_size": (224, 224),
}

# Auto-detect device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("🔧 Preprocessing configuration:")
for key, value in PREPROCESSING_CONFIG.items():
    print(f"  • {key}: {value}")
print(f"  • device: {device}")